# 6.6. 卷积神经网络（LeNet）

LeNet是最早发布的卷积神经网络之一，由Yann LeCun在1998年提出，用于手写数字识别。虽然LeNet在现代标准下显得简单，但它展示了卷积神经网络的基本组件：
- 卷积层
- 池化层  
- 全连接层

In [ ]:
import tensorflow as tf
import numpy as np
import time

## 6.6.1. LeNet架构

LeNet包含两个部分：
1. **卷积编码器**：由两个卷积层组成
2. **全连接层块**：由三个全连接层组成

In [ ]:
# LeNet模型
def LeNet():
    return tf.keras.models.Sequential([
        # 卷积层1: 6个5×5的卷积核，使用sigmoid激活
        tf.keras.layers.Conv2D(filters=6, kernel_size=5, activation='sigmoid',
                               padding='same'),
        # 平均池化层: 2×2窗口，步幅2
        tf.keras.layers.AvgPool2D(pool_size=2, strides=2),
        # 卷积层2: 16个5×5的卷积核，使用sigmoid激活
        tf.keras.layers.Conv2D(filters=16, kernel_size=5, activation='sigmoid'),
        # 平均池化层: 2×2窗口，步幅2
        tf.keras.layers.AvgPool2D(pool_size=2, strides=2),
        # 展平层
        tf.keras.layers.Flatten(),
        # 全连接层1: 120个输出
        tf.keras.layers.Dense(120, activation='sigmoid'),
        # 全连接层2: 84个输出
        tf.keras.layers.Dense(84, activation='sigmoid'),
        # 输出层: 10个输出（对应10个类别）
        tf.keras.layers.Dense(10)
    ])

In [ ]:
# 创建模型并查看架构
net = LeNet()
X = tf.random.uniform((1, 28, 28, 1))
Y = net(X)
print("LeNet模型架构：\n")
net.summary()

## 6.6.2. 数据流动可视化

让我们看看数据如何在LeNet中流动：

In [ ]:
# 追踪每一层的输出形状
X = tf.random.uniform((1, 28, 28, 1))
print("输入形状:", X.shape)

for i, layer in enumerate(net.layers):
    X = layer(X)
    print(f'{layer.__class__.__name__} 输出形状:\t', X.shape)

## 6.6.3. 在Fashion-MNIST上训练

In [ ]:
# 加载数据
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = np.expand_dims(x_train.astype(np.float32) / 255.0, axis=-1)
x_test = np.expand_dims(x_test.astype(np.float32) / 255.0, axis=-1)

print(f"训练集形状: {x_train.shape}")
print(f"测试集形状: {x_test.shape}")

In [ ]:
# 创建数据集
batch_size = 256
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(batch_size)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_dataset = test_dataset.batch(batch_size)

In [ ]:
# 定义训练函数
def train_epoch(net, train_iter, loss, optimizer):
    """训练一个epoch"""
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0
    
    for X, y in train_iter:
        with tf.GradientTape() as tape:
            y_hat = net(X, training=True)
            l = loss(y, y_hat)
        
        grads = tape.gradient(l, net.trainable_variables)
        optimizer.apply_gradients(zip(grads, net.trainable_variables))
        
        total_loss += l
        total_acc += tf.reduce_sum(
            tf.cast(tf.argmax(y_hat, axis=1) == y, dtype=tf.float32))
        num_batches += 1
    
    return total_loss / num_batches, total_acc / len(x_train)

In [ ]:
# 定义评估函数
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上的准确率"""
    total_acc = 0.0
    num_examples = 0
    
    for X, y in data_iter:
        y_hat = net(X, training=False)
        total_acc += tf.reduce_sum(
            tf.cast(tf.argmax(y_hat, axis=1) == y, dtype=tf.float32))
        num_examples += X.shape[0]
    
    return total_acc / num_examples

In [ ]:
# 训练模型
net = LeNet()
lr = 0.9
num_epochs = 10
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr)

print("开始训练...\n")
for epoch in range(num_epochs):
    start = time.time()
    train_loss, train_acc = train_epoch(net, train_dataset, loss, optimizer)
    test_acc = evaluate_accuracy(net, test_dataset)
    
    print(f'epoch {epoch + 1}, '
          f'loss {float(train_loss):.3f}, '
          f'train acc {float(train_acc):.3f}, '
          f'test acc {float(test_acc):.3f}, '
          f'time {time.time() - start:.1f}s')

print("\n训练完成！")

## 6.6.4. 模型预测示例

In [ ]:
# Fashion-MNIST类别标签
text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
               'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']

# 预测前10个测试样本
X_sample = x_test[:10]
y_sample = y_test[:10]
y_pred = tf.argmax(net(X_sample, training=False), axis=1)

print("预测结果对比：\n")
for i in range(10):
    true_label = text_labels[y_sample[i]]
    pred_label = text_labels[int(y_pred[i])]
    result = "✓" if y_sample[i] == y_pred[i] else "✗"
    print(f"样本{i+1}: 真实={true_label:12s} 预测={pred_label:12s} {result}")

## 小结

1. **LeNet架构**：
   - 2个卷积层（带池化）提取特征
   - 3个全连接层进行分类
   - 展示了CNN的基本设计模式

2. **关键特点**：
   - 交替使用卷积层和池化层
   - 逐层增加通道数
   - 逐层减小空间维度
   - 最后用全连接层进行分类

3. **现代改进**：
   - 使用ReLU替代Sigmoid
   - 使用最大池化替代平均池化
   - 添加批量归一化
   - 使用Dropout防止过拟合

4. **历史意义**：LeNet证明了卷积神经网络在图像识别任务上的有效性，为后续的CNN发展奠定了基础。